In [104]:
import pandas as pd
import seaborn as sns

taxis = sns.load_dataset('taxis')
df = taxis.copy()
df.shape
df.head()


,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


On constate 14 colonnes et 6433 lignes, donc 6433 courses pour ce dataset.

In [105]:
print(df.dtypes)

pickup             datetime64[us]
dropoff            datetime64[us]
passengers                  int64
distance                  float64
fare                      float64
tip                       float64
tolls                     float64
total                     float64
color                         str
payment                       str
pickup_zone                   str
dropoff_zone                  str
pickup_borough                str
dropoff_borough               str
dtype: object


Ici on remarque que "pickup" et "dropoff" sont deja en datatime.

In [106]:
print(f"Il y a {df['pickup'].dt.date.nunique()} jours différents dans le dataset.\n")

print("CA journaliers :")
display(
    df.groupby(df['pickup'].dt.date).agg(
        nb_courses=('fare', 'count'),
        ca=('fare', 'sum')
    ).reset_index()
)

print(f"\nCA Moyen par jour : {df['fare'].sum() / df['pickup'].dt.date.nunique():.2f} €")

ca_par_jour = (
    df.set_index("pickup")
         .resample("D")["total"]
         .sum()
)



Il y a 32 jours différents dans le dataset.

CA journaliers :


,pickup,nb_courses,ca
0,2019-02-28,1,5.00
1,2019-03-01,241,2946.97
2,2019-03-02,198,2358.00
3,2019-03-03,169,2187.89
4,2019-03-04,171,2335.74
5,2019-03-05,228,3137.97
6,2019-03-06,257,3269.70
7,2019-03-07,218,2898.64
8,2019-03-08,235,3168.50
9,2019-03-09,204,2741.05



CA Moyen par jour : 2631.71 €


Pour un nombre total de 6433 courses, nous avons un CA moyen par jour de 2631.71€

In [107]:
jour_plus_charge = df["pickup"].dt.date.value_counts().idxmax()

print(f"Le jour le plus chargé est le : {jour_plus_charge}")

heure_plus_charge = (df[df["pickup"].dt.date == jour_plus_charge]["pickup"].dt.hour.value_counts().idxmax())
print(f"L'heure la plus chargée ce jour-là est : {heure_plus_charge}h")

Le jour le plus chargé est le : 2019-03-14
L'heure la plus chargée ce jour-là est : 18h


In [108]:
pourboire_par_date = df.groupby(
    df["pickup"].dt.date
)["tip"].mean()

date_max = pourboire_par_date.idxmax()

pourboire_moyen_par_jour = df.groupby(
    df["pickup"].dt.day_name()
)["tip"].mean()

display(
    pourboire_moyen_par_jour
    .sort_values(ascending=False)
    .round(2)
    .reset_index(name="pourboire_moyen")
    )

print(
    f"Date avec le pourboire moyen le plus élevé : "
    f"{date_max.strftime('%A')} {date_max} "
    f"({pourboire_par_date.max():.2f}$)"
)

print(
    f"Jour de la semaine avec le pourboire moyen le plus élevé : "
    f"{pourboire_moyen_par_jour.idxmax()} "
    f"({pourboire_moyen_par_jour.max():.2f}$)"
)

,pickup,pourboire_moyen
0,Wednesday,2.09
1,Thursday,2.06
2,Monday,2.03
3,Tuesday,2.00
4,Friday,1.98
5,Sunday,1.91
6,Saturday,1.82


Date avec le pourboire moyen le plus élevé : Thursday 2019-03-21 (2.34$)
Jour de la semaine avec le pourboire moyen le plus élevé : Wednesday (2.09$)


In [109]:
df["duree_course"] = (
    df["dropoff"] - df["pickup"]
).dt.total_seconds() / 60

display(df["duree_course"].describe().round(2))

print(f"Durée minimale : {df['duree_course'].min():.2f} min")
print(f"Durée maximale : {df['duree_course'].max():.2f} min")

count    6433.00
mean       14.35
std        11.64
min         0.00
25%         6.50
50%        10.90
75%        18.52
max       107.67
Name: duree_course, dtype: float64

Durée minimale : 0.00 min
Durée maximale : 107.67 min


In [110]:
Q1 = df["duree_course"].quantile(0.25)
Q3 = df["duree_course"].quantile(0.75)
IQR = Q3 - Q1

borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

df = df[
    (df["duree_course"] >= borne_inf)
    & (df["duree_course"] <= borne_sup)
]

df["duree_min"] = (
    df["dropoff"] - df["pickup"]
).dt.total_seconds() / 60

display(df["duree_min"].describe().round(2))

print(f"Durée minimale : {df['duree_min'].min():.2f} min")
print(f"Durée maximale : {df['duree_min'].max():.2f} min")


count    6075.00
mean       12.31
std         7.86
min         0.00
25%         6.32
50%        10.28
75%        16.90
max        36.52
Name: duree_min, dtype: float64

Durée minimale : 0.00 min
Durée maximale : 36.52 min
